# 0. Problem
## 1204. Last Person to Fit in the Bus — Medium
Passengers board by `turn`. The bus limit is 1000 kg. Return the last person's name whose cumulative weight does not exceed 1000.

Official: https://leetcode.com/problems/last-person-to-fit-in-the-bus/

# 1. Setup

In [ ]:
import pandas as pd
queue_rows=[(5,"Alice",250,1),(4,"Bob",175,5),(3,"Alex",350,2),(6,"John Cena",400,3),(1,"Winston",500,6),(2,"Marie",200,4)]
queue_pd=pd.DataFrame(queue_rows,columns=["person_id","person_name","weight","turn"])
queue_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
queue_spark=spark.createDataFrame(queue_rows,["person_id","person_name","weight","turn"])
queue_spark.createOrReplaceTempView("Queue")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH x AS (
  SELECT person_name,turn,
         SUM(weight) OVER(ORDER BY turn ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS total_weight
  FROM Queue
)
SELECT person_name
FROM x
WHERE total_weight<=1000
ORDER BY turn DESC
LIMIT 1
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
work_pd=queue_pd.sort_values("turn").copy()
work_pd["total_weight"]=work_pd["weight"].cumsum()
result_pd=work_pd.loc[work_pd["total_weight"]<=1000,["person_name"]].tail(1).reset_index(drop=True)
result_pd

# 4. PySpark Solution

In [ ]:
w=Window.orderBy("turn").rowsBetween(Window.unboundedPreceding,Window.currentRow)
result_spark=(queue_spark.withColumn("total_weight",F.sum("weight").over(w)).filter(F.col("total_weight")<=1000).orderBy(F.desc("turn")).select("person_name").limit(1))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| running sum | `SUM() OVER` | `.cumsum()` | Window `F.sum()` |
| last valid row | desc + `LIMIT 1` | `.tail(1)` | desc + `.limit(1)` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Queue

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: queue_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: queue_spark